# L14: 向量数据库基础

> 理解如何让 AI「记住」大量知识并快速检索

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.rag.embeddings import EmbeddingModel
from backend.rag.vectorstore import VectorStore
print("✓ 模块导入成功")

## 14.1 什么是向量数据库？

### 从搜索到语义搜索

```
传统搜索（关键词匹配）:
  查询: "苹果手机"
  匹配: 包含「苹果」「手机」的文档
  
语义搜索（向量相似度）:
  查询: "苹果手机"
  匹配: iPhone、苹果出产的智能手机、...
```

### 向量数据库的核心概念

| 概念 | 说明 |
|------|------|
| **向量 (Embedding)** | 文本的数值表示，捕捉语义信息 |
| **相似度** | 两个向量之间的接近程度 |
| **索引** | 加速向量检索的数据结构 |
| **元数据** | 与向量关联的额外信息 |

## 14.2 文本向量化 (Embeddings)

### 什么是 Embedding？

**Embedding** = 将文本映射到高维数值向量，相似含义的文本在向量空间中距离更近。

```
"猫"  → [0.2, -0.5, 0.8, ...]  (384 维向量)
"狗"  → [0.3, -0.4, 0.7, ...]  (与「猫」距离近)
"汽车" → [-0.8, 0.6, -0.2, ...]  (与「猫」距离远)
```

In [ ]:
# 查看项目中的 Embedding 实现
import inspect
from backend.rag.embeddings import EmbeddingModel

print("=== EmbeddingModel 源码 ===\n")
print(inspect.getsource(EmbeddingModel))

### 常见 Embedding 模型

| 模型 | 维度 | 特点 |
|------|------|------|
| **text-embedding-ada-002** | 1536 | OpenAI，英文优化 |
| **text-embedding-3-small** | 1536 | OpenAI，性价比高 |
| **bge-large-zh** | 1024 | BAAI，中文优化 |
| **m3e-base** | 768 | 中文通用 |
| **all-MiniLM-L6-v2** | 384 | 轻量级，多语言 |

### 选择 Embedding 模型的标准

- **语言**：中文优先选 bge/m3e
- **维度**：高维度更准确但成本更高
- **部署**：本地部署 vs API 调用
- **场景**：检索密集型选轻量级

## 14.3 相似度计算

### 余弦相似度 (Cosine Similarity)

最常用的向量相似度计算方法。

```
cosine_similarity(A, B) = (A · B) / (||A|| × ||B||)

值域: [-1, 1]
  1   = 完全相同方向
  0   = 正交（无关）
  -1  = 完全相反
```

In [ ]:
import numpy as np
from typing import List

def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    """计算余弦相似度"""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    if norm1 == 0 or norm2 == 0:
        return 0.0
    
    return dot_product / (norm1 * norm2)

# 测试相似度计算
def demo_similarity():
    """演示相似度计算"""
    
    # 模拟向量（简化版）
    cat = [0.9, 0.1, 0.8]    # 猫的特征向量
    dog = [0.8, 0.2, 0.7]    # 狗的特征向量（与猫相似）
    car = [0.1, 0.9, 0.2]    # 汽车的特征向量（与猫不相似）
    
    print("=== 余弦相似度测试 ===\n")
    
    print(f"猫 vs 狗: {cosine_similarity(cat, dog):.4f}")
    print(f"猫 vs 汽车: {cosine_similarity(cat, car):.4f}")
    print(f"狗 vs 汽车: {cosine_similarity(dog, car):.4f}")
    
    print("\n解释:")
    print("- 猫和狗的相似度高，因为都是动物")
    print("- 猫和汽车的相似度低，因为完全不同")

demo_similarity()

## 14.4 向量索引结构

### 为什么需要索引？

```
暴力搜索:
  100万向量 × 每次查询 = 100万次计算
  
索引搜索:
  先用索引缩小范围，再计算相似度
  只需要计算几百次
```

### 常见索引类型

| 索引类型 | 原理 | 优点 | 缺点 |
|----------|------|------|------|
| **FLAT** | 暴力搜索 | 100% 召回率 | 慢 |
| **IVF** | 倒排文件 | 平衡 | 需要训练 |
| **HNSW** | 图索引 | 最快 | 内存占用高 |
| **PQ** | 乘积量化 | 压缩 | 精度损失 |

## 14.5 项目中的向量存储实现

In [ ]:
import inspect
from backend.rag.vectorstore import VectorStore

print("=== VectorStore 源码 ===\n")
print(inspect.getsource(VectorStore))

## 14.6 实战：构建简单的向量存储

In [ ]:
import numpy as np
from typing import List, Dict, Any, Tuple
import uuid

class SimpleVectorStore:
    """简单的向量存储实现"""
    
    def __init__(self, dimension: int = 384):
        self.dimension = dimension
        self.vectors: Dict[str, np.ndarray] = {}  # id -> vector
        self.metadata: Dict[str, Dict[str, Any]] = {}  # id -> metadata
    
    def add(
        self, 
        vector: List[float], 
        metadata: Dict[str, Any] = None,
        doc_id: str = None
    ) -> str:
        """添加向量"""
        if doc_id is None:
            doc_id = str(uuid.uuid4())
        
        # 转换为 numpy 数组
        arr = np.array(vector, dtype=np.float32)
        
        # 验证维度
        if len(arr) != self.dimension:
            raise ValueError(f"向量维度不匹配: 期望 {self.dimension}, 实际 {len(arr)}")
        
        self.vectors[doc_id] = arr
        self.metadata[doc_id] = metadata or {}
        
        return doc_id
    
    def search(
        self, 
        query_vector: List[float], 
        top_k: int = 5
    ) -> List[Tuple[str, float, Dict[str, Any]]]:
        """搜索最相似的向量"""
        if not self.vectors:
            return []
        
        query = np.array(query_vector, dtype=np.float32)
        
        # 计算与所有向量的相似度
        similarities = []
        for doc_id, vec in self.vectors.items():
            sim = cosine_similarity(query, vec)
            similarities.append((doc_id, sim, self.metadata[doc_id]))
        
        # 按相似度排序
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        return similarities[:top_k]
    
    def delete(self, doc_id: str):
        """删除向量"""
        if doc_id in self.vectors:
            del self.vectors[doc_id]
            del self.metadata[doc_id]
    
    def get(self, doc_id: str) -> Optional[Tuple[np.ndarray, Dict[str, Any]]]:
        """获取向量"""
        if doc_id in self.vectors:
            return (self.vectors[doc_id], self.metadata[doc_id])
        return None
    
    def __len__(self):
        return len(self.vectors)

# 测试向量存储
async def test_vector_store():
    store = SimpleVectorStore(dimension=3)  # 使用 3 维便于演示
    
    print("=== 向量存储测试 ===\n")
    
    # 添加向量（模拟）
    documents = [
        {"text": "猫是可爱的宠物", "vector": [0.9, 0.1, 0.8], "category": "动物"},
        {"text": "狗是人类最好的朋友", "vector": [0.8, 0.2, 0.7], "category": "动物"},
        {"text": "汽车是交通工具", "vector": [0.1, 0.9, 0.2], "category": "交通工具"},
        {"text": "飞机可以飞", "vector": [0.2, 0.8, 0.1], "category": "交通工具"},
    ]
    
    print("添加文档:")
    for doc in documents:
        doc_id = store.add(
            vector=doc["vector"],
            metadata={"text": doc["text"], "category": doc["category"]}
        )
        print(f"  - {doc['text']}")
    
    print(f"\n总共存储了 {len(store)} 个向量\n")
    
    # 搜索测试
    queries = [
        {"text": "关于宠物的", "vector": [0.85, 0.15, 0.75]},  # 接近猫/狗
        {"text": "关于交通的", "vector": [0.15, 0.85, 0.15]},  # 接近汽车/飞机
    ]
    
    for q in queries:
        print(f"查询: {q['text']}")
        results = store.search(q["vector"], top_k=2)
        for doc_id, score, meta in results:
            print(f"  - {meta['text']} (相似度: {score:.4f})")
        print()

await test_vector_store()

## 14.7 向量数据库选型

### 主流向量数据库对比

| 数据库 | 特点 | 适用场景 |
|--------|------|----------|
| **ChromaDB** | 轻量级，易集成 | 原型开发、本地存储 |
| **Pinecone** | 托管服务，易扩展 | 生产环境、无需运维 |
| **Milvus** | 开源，功能丰富 | 自建大规模部署 |
| **Weaviate** | GraphQL 支持 | 需要 GraphQL 的场景 |
| **Qdrant** | Rust 实现，高性能 | 性能敏感场景 |
| **Faiss** | Facebook 出品，纯库 | 需要完全控制 |

### 本项目使用 ChromaDB

- Python 原生，易于集成
- 支持持久化存储
- 内置多种 embedding 模型
- 简单易用的 API

## 14.8 常见面试问题

**Q: 向量数据库和传统数据库有什么区别？**
- A:
  | 特性 | 传统数据库 | 向量数据库 |
  |------|-----------|------------|
  | 查询方式 | 精确匹配 | 相似度搜索 |
  | 索引类型 | B-Tree, Hash | HNSW, IVF |
  | 搜索类型 | 关键词、过滤 | 语义相似 |
  | 用途 | 结构化数据 | 非结构化语义检索 |

**Q: 什么是向量维度？如何选择？**
- A:
  - **维度** = 向量的长度（如 384, 768, 1536）
  - 高维度：更精确，但计算和存储成本高
  - 低维度：更快，但可能损失精度
  - 选择建议：中文选 768-1024，英文可选 1536

**Q: HNSW 索引为什么快？**
- A:
  1. **图结构**：构建多层图，类似跳表
  2. **贪心搜索**：从顶层开始，快速定位到目标区域
  3. **时间复杂度**：O(log N) 而非 O(N)
  4. **权衡**：内存占用高，但查询速度极快

**Q: 如何评估向量检索质量？**
- A:
  1. **召回率**：相关文档被检索到的比例
  2. **精确率**：检索结果中相关文档的比例
  3. **MRR**：第一个相关结果的排名倒数
  4. **NDCG**：考虑排序位置的评价指标

**Q: RAG 中向量数据库的作用是什么？**
- A:
  1. 存储文档的向量表示
  2. 根据问题快速检索相关文档
  3. 将检索结果作为上下文提供给 LLM
  4. 实现「知识增强」的问答

---

## 总结

本课程学习了向量数据库的基础知识：

| 知识点 | 说明 |
|--------|------|
| **Embedding** | 文本的数值表示，捕捉语义 |
| **余弦相似度** | 衡量向量相似程度的标准方法 |
| **向量索引** | HNSW/IVF 加速检索 |
| **向量数据库** | 专门存储和检索向量的数据库 |
| **ChromaDB** | 本项目使用的向量数据库 |
| **选型标准** | 场景、规模、性能、成本 |

**下一步**: L15 将学习长期记忆的完整实现！